# 5일차 미니 프로젝트 - 시작 코드 생성

이 노트북을 실행하면 미니 프로젝트용 `mini_project/app/` 트리가 자동으로 생성됩니다.

5-2 의 Router / Service / Client 분리 위에 5-4 의 로깅 / 인메모리 캐싱을 얹은 형태입니다.

**학생이 채울 자리는 두 곳뿐입니다.**

- `app/prompts/system_prompt.py` — 도메인별 System Prompt 한 줄
- `app/prompts/demo_questions.py` — 데모 질문 5개

나머지는 그대로 실행만 하면 됩니다.

## 1. 환경 준비

5일차 실습과 동일합니다. 같은 패키지(`fastapi`, `uvicorn`, `httpx`, `openai`, `pydantic`) 를 씁니다.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

Path("mini_project").mkdir(exist_ok=True)
load_dotenv("mini_project/.env")

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

# config.py가 읽을 수 있도록 mini_project/.env 에 저장
Path("mini_project/.env").write_text(
    f"MLAPI_BASE_URL={base_url}\n"
    f"MLAPI_API_KEY={api_key}\n"
    f"MLAPI_MODEL={model_name}\n",
    encoding="utf-8",
)
Path("mini_project/.gitignore").write_text(".env\n__pycache__/\n", encoding="utf-8")

PORT = 8003                 # 서버 포트. 충돌나면 8001 등으로 변경
SERVER_BASE = f"http://localhost:{PORT}"

print("모델명:", model_name, "/ PORT:", PORT)
print("server base:", SERVER_BASE)
print("mini_project/.env 저장 완료 — config.py가 이 파일에서 키를 읽습니다")

## 2. 프로젝트 폴더 / 빈 파일 자동 생성

5-2 와 비교해 두 가지가 추가됐습니다.

- `core/logging_config.py` — 로깅 설정
- `services/cache.py` — 인메모리 캐시
- `prompts/` — 학생이 채울 자리 (System Prompt + 데모 질문)

```
mini_project/
  app/
    __init__.py
    main.py
    core/
      __init__.py
      config.py
      logging_config.py
    schemas/
      __init__.py
      chat.py
    clients/
      __init__.py
      openai_client.py
    services/
      __init__.py
      chat_service.py
      cache.py
    prompts/
      __init__.py
      system_prompt.py
      demo_questions.py
    routers/
      __init__.py
      chat.py
```

In [ ]:
from pathlib import Path

ROOT = Path("mini_project/app")
SUBDIRS = ["core", "schemas", "clients", "services", "prompts", "routers"]

ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / "__init__.py").touch()
for name in SUBDIRS:
    (ROOT / name).mkdir(exist_ok=True)
    (ROOT / name / "__init__.py").touch()

print("폴더 / __init__.py 생성 완료")
for p in sorted(ROOT.rglob("*")):
    print("  -", p)

## 3. app/core/config.py

5-2 와 동일. 환경 변수만 모아둡니다.

In [ ]:
from pathlib import Path

code = '''import os
from dotenv import load_dotenv

load_dotenv()

API_KEY  = os.getenv("MLAPI_API_KEY")
MODEL    = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")
BASE_URL = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")

if not API_KEY:
    raise RuntimeError("MLAPI_API_KEY가 설정되어 있지 않습니다. .env 파일을 확인하세요.")
'''

Path("mini_project/app/core/config.py").write_text(code, encoding="utf-8")
print("app/core/config.py 작성 완료")

## 4. app/core/logging_config.py — 로깅 설정

`setup_logging()` 한 번 호출로 포맷·레벨이 일괄 적용됩니다.

- 레벨: INFO
- 포맷: 시간 / 레벨 / 로거 이름 / 메시지
- 핸들러: 콘솔 (서버 터미널에 그대로 흐름)

In [ ]:
from pathlib import Path

code = '''
import logging


def setup_logging(level: int = logging.INFO) -> None:
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
'''

Path("mini_project/app/core/logging_config.py").write_text(code, encoding="utf-8")
print("app/core/logging_config.py 작성 완료")

## 5. app/schemas/chat.py — 요청 모델

5-2 와 동일.

In [ ]:
from pathlib import Path

code = '''
from typing import Optional
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1)
    model: Optional[str] = Field(None)
    temperature: Optional[float] = Field(None, ge=0.0, le=2.0)
'''

Path("mini_project/app/schemas/chat.py").write_text(code, encoding="utf-8")
print("app/schemas/chat.py 작성 완료")

## 6. app/services/cache.py — 인메모리 캐시

메시지 + 모델 + temperature 를 합쳐 SHA256 해시로 캐시 키를 만듭니다. 저장소는 프로세스 안의 dict 이고, 서버 재시작 시 비워집니다.

- `make_key(messages, model, temperature)` — 키 생성
- `get(key)` / `set(key, value)` — 단순 조회/저장

In [ ]:
from pathlib import Path

code = '''
import hashlib
import json
from typing import List, Dict, Optional

_store: Dict[str, str] = {}


def make_key(
    messages: List[Dict[str, str]],
    model: Optional[str],
    temperature: Optional[float],
) -> str:
    payload = json.dumps(
        {"messages": messages, "model": model, "temperature": temperature},
        ensure_ascii=False,
        sort_keys=True,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def get(key: str) -> Optional[str]:
    return _store.get(key)


def set(key: str, value: str) -> None:
    _store[key] = value


def size() -> int:
    return len(_store)
'''

Path("mini_project/app/services/cache.py").write_text(code, encoding="utf-8")
print("app/services/cache.py 작성 완료")

## 7. app/clients/openai_client.py — LLM 호출 + 로깅

5-2 와 거의 같지만 호출 전후로 로깅이 추가되어 있습니다.

- 호출 시작 시: 사용 모델 / temperature
- 호출 종료 시: 토큰 piece 누적 개수

In [ ]:
from pathlib import Path

code = '''
import logging
from typing import AsyncIterator, List, Dict, Optional
from openai import AsyncOpenAI
from app.core.config import API_KEY, MODEL, BASE_URL

logger = logging.getLogger(__name__)


def get_client() -> AsyncOpenAI:
    if BASE_URL:
        return AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)
    return AsyncOpenAI(api_key=API_KEY)


_client = get_client()


async def stream_chat(
    messages: List[Dict[str, str]],
    model: Optional[str] = None,
    temperature: Optional[float] = None,
    timeout: float = 10.0,
) -> AsyncIterator[str]:
    use_model = model or MODEL
    use_temp = temperature if temperature is not None else 1.0
    logger.info("llm call start | model=%s temperature=%.2f", use_model, use_temp)

    stream = await _client.chat.completions.create(
        model=use_model,
        messages=messages,
        temperature=use_temp,
        stream=True,
        timeout=timeout,
    )
    count = 0
    async for chunk in stream:
        if not chunk.choices:
            continue
        piece = getattr(chunk.choices[0].delta, "content", None)
        if not piece:
            continue
        count += 1
        yield piece
    logger.info("llm call end | pieces=%d", count)
'''

Path("mini_project/app/clients/openai_client.py").write_text(code, encoding="utf-8")
print("app/clients/openai_client.py 작성 완료")

## 8. app/services/chat_service.py — 캐시 + 안전 패턴

5-2 의 `stream_chat_safe` 위에 캐시 조회/저장을 얹었습니다.

- 호출 시작 시 캐시 키 생성 → 히트면 저장된 본문을 한 번에 yield (`[DONE]` 마커 포함)
- 미스면 LLM 호출하며 토큰을 yield 하고, 동시에 누적해두었다가 종료 시점에 캐시에 저장
- 본문 끝 마커: `[DONE]` / `[EMPTY]` / `[ERROR]` 셋 중 하나로 유지
- 응답 시간(ms) 과 캐시 히트 여부를 로깅

In [ ]:
from pathlib import Path

code = '''
import asyncio
import logging
import time
from typing import AsyncIterator
from app.schemas.chat import ChatRequest
from app.clients.openai_client import stream_chat
from app.prompts.system_prompt import SYSTEM_PROMPT
from app.services import cache

logger = logging.getLogger(__name__)


def _build_messages(req: ChatRequest):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": req.message},
    ]


async def stream_chat_safe(req: ChatRequest) -> AsyncIterator[str]:
    started = time.time()
    messages = _build_messages(req)
    key = cache.make_key(messages, req.model, req.temperature)
    hit = cache.get(key)

    if hit is not None:
        logger.info("cache HIT | key=%s | size=%d", key[:8], cache.size())
        yield hit
        yield "\\n[DONE]"
        logger.info("request done (cache) | elapsed=%dms", int((time.time() - started) * 1000))
        return

    logger.info("cache MISS | key=%s", key[:8])
    produced = False
    buffer = []
    try:
        async for piece in stream_chat(
            messages=messages,
            model=req.model,
            temperature=req.temperature,
        ):
            produced = True
            buffer.append(piece)
            yield piece
        if produced:
            full = "".join(buffer)
            cache.set(key, full)
            logger.info("cache STORE | key=%s | size=%d", key[:8], cache.size())
            yield "\\n[DONE]"
        else:
            yield "[EMPTY]"
            yield "\\n"
    except asyncio.CancelledError:
        logger.warning("cancelled by client")
        raise
    except Exception as e:
        logger.error("llm error | %s: %s", type(e).__name__, e)
        yield f"[ERROR] {type(e).__name__}: {e}"
        yield "\\n"
    finally:
        logger.info("request done | elapsed=%dms", int((time.time() - started) * 1000))
'''

Path("mini_project/app/services/chat_service.py").write_text(code, encoding="utf-8")
print("app/services/chat_service.py 작성 완료")

## 9. app/prompts/system_prompt.py — 학생이 채울 자리 ①

도메인에 맞는 System Prompt 한 줄(또는 몇 줄) 을 작성하세요. 5-3 의 권장 순서(역할 → 톤 → 언어 → 제약) 를 참고하면 좋습니다.

지금은 안내 문구가 들어 있는 placeholder 입니다.

In [ ]:
from pathlib import Path

code = '''
# 도메인에 맞는 System Prompt 를 작성하세요.
# 권장 순서: 역할 → 톤 → 언어 → 제약

# 예시:
SYSTEM_PROMPT = (
    "너는 생활 속 법률 문제를 일반인에게 쉽게 풀어 설명하는 상담가야. "
    "전문 용어는 한 줄 풀이를 곁들이고, 한국어로 친절하게 3문장 이내로 답해. "
    "법적 판단이 필요한 사안에는 '실제 사안은 전문가 상담을 권합니다'를 덧붙여. "
    "날씨·음식·잡담처럼 법률과 명백히 무관한 질문에만 '법률 관련 질문만 도와드릴 수 있어요'라고 답하고, 법률 관련 질문에는 정상적으로 답해."
)


# SYSTEM_PROMPT = (
#     " "
#     " "
#     " "
#     " "
#     " "
# )

'''

Path("mini_project/app/prompts/system_prompt.py").write_text(code, encoding="utf-8")
print("app/prompts/system_prompt.py 작성 완료 (placeholder)")

## 10. app/prompts/demo_questions.py — 학생이 채울 자리 ②

발표용 데모 질문 5개를 작성하세요. System Prompt 와 잘 맞는, 도메인 성격이 드러나는 질문이 좋습니다.

지금은 안내 문구가 들어 있는 placeholder 입니다.

In [ ]:
from pathlib import Path

code = '''
# 발표용 데모 질문 5개를 작성하세요.
# - System Prompt 가 잘 드러나는 질문
# - 짧고 답이 한눈에 들어오는 질문
# - 너무 비슷하지 않은 질문 다섯 개

# 예시
# DEMO_QUESTIONS = [
#     "전세 계약할 때 가장 먼저 확인해야 할 게 뭐야?",
#     "근저당권이랑 저당권 차이를 한 문장으로 설명해줘",
#     "임대인이 보증금을 안 돌려주면 어떻게 해야 해?",
#     "내가 직접 쓴 차용증도 법적 효력이 있어?",
#     "오늘 점심 메뉴 좀 추천해줘",
# ]

DEMO_QUESTIONS = [
    "여기에 질문 1",
    "여기에 질문 2",
    "여기에 질문 3",
    "여기에 질문 4",
    "여기에 질문 5",
]

'''

Path("mini_project/app/prompts/demo_questions.py").write_text(code, encoding="utf-8")
print("app/prompts/demo_questions.py 작성 완료 (placeholder)")

## 11. app/routers/chat.py — HTTP 입출력

5-2 와 동일.

In [ ]:
from pathlib import Path

code = '''
from fastapi import APIRouter
from fastapi.responses import StreamingResponse
from app.schemas.chat import ChatRequest
from app.services.chat_service import stream_chat_safe

router = APIRouter()

@router.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    return StreamingResponse(stream_chat_safe(req), media_type="text/plain")
'''

Path("mini_project/app/routers/chat.py").write_text(code, encoding="utf-8")
print("app/routers/chat.py 작성 완료")

## 12. app/main.py — 앱 조립 + 로깅 셋업

앱 생성 직후 `setup_logging()` 을 한 번 호출합니다. 이후 모든 모듈에서 `logging.getLogger(__name__)` 만 쓰면 자동으로 같은 포맷으로 흐릅니다.

In [ ]:
from pathlib import Path

code = '''
from fastapi import FastAPI
from app.core.logging_config import setup_logging
from app.routers.chat import router as chat_router

setup_logging()

app = FastAPI()
app.include_router(chat_router)

@app.get("/health")
def health():
    return {"ok": True}
'''

Path("mini_project/app/main.py").write_text(code, encoding="utf-8")
print("app/main.py 작성 완료")

## 13. 서버 띄우기

터미널에서 아래 셀이 출력하는 명령어를 그대로 붙여넣어 실행하세요.

- 작업 디렉터리는 `mini_project/` 입니다. 노트북과 같은 폴더에서 `cd mini_project` 후 실행하세요.
- `--reload` 로 어떤 파일을 고쳐도 자동 재시작됩니다.
- 포트 충돌 시 1 단계 `PORT` 만 바꾸고 3, 13 단계 셀 재실행.

In [ ]:
print("cd mini_project")
print(f"uvicorn app.main:app --reload --port {PORT}")
print()
print("서버 주소:", SERVER_BASE)

In [ ]:
import httpx

r = httpx.get(f"{SERVER_BASE}/health", timeout=5.0)
print(r.status_code, r.json())

## 14. 정상 호출 확인

System Prompt 와 데모 질문을 채우기 전이라도, placeholder 상태로 한 번 호출해 보고 서버가 정상 동작하는지 확인합니다.

- status 200, 토큰이 한 토큰씩 흘러오는지
- 본문 끝이 `[DONE]` 으로 끝나는지
- 서버 터미널 로그에 `llm call start`, `llm call end`, `request done | elapsed=...ms` 가 보이는지

In [ ]:
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream",
        json={"message": "안녕? 한 줄로 자기소개 해줘."},
    ) as r:
        print("status:", r.status_code)
        body = ""
        for piece in r.iter_text():
            body += piece
            print(piece, end="", flush=True)
print("\n--- end ---")
assert body.rstrip().endswith("[DONE]"), "정상 응답은 [DONE] 으로 끝나야 합니다."

## 15. 캐시 동작 확인

같은 질문을 두 번 보내봅니다.

- 첫 번째 호출: 서버 로그에 `cache MISS` → `llm call start/end` → `cache STORE` → `request done`
- 두 번째 호출: 서버 로그에 `cache HIT` → `request done (cache)` (LLM 호출 없음)
- 두 번째는 응답이 훨씬 빠르게 한 번에 도착합니다.
- 본문 마지막 줄은 두 호출 모두 `[DONE]` 으로 같습니다.

In [ ]:
import time
import httpx

QUESTION = "오늘 점심 메뉴 한 가지만 추천해줘."

def call_once(label: str):
    t0 = time.time()
    with httpx.Client(timeout=60.0) as c:
        with c.stream(
            "POST",
            f"{SERVER_BASE}/chat/stream",
            json={"message": QUESTION},
        ) as r:
            body = "".join(r.iter_text())
    elapsed = int((time.time() - t0) * 1000)
    print(f"[{label}] elapsed={elapsed}ms, end={body.rstrip()[-10:]!r}")
    return body

b1 = call_once("1st (miss)")
b2 = call_once("2nd (hit)")

assert b1.rstrip().endswith("[DONE]") and b2.rstrip().endswith("[DONE]")
print("\n서버 터미널에서 1st 에는 cache MISS / STORE, 2nd 에는 cache HIT 로그가 보이는지 확인하세요.")

## 16. 데모 질문 5개 일괄 호출

`app/prompts/demo_questions.py` 에 적힌 5개 질문을 차례로 호출합니다. 발표 자료에 그대로 캡처해 쓰면 됩니다.

- 이 셀을 실행하려면 노트북에서 `mini_project` 가 import 가능해야 합니다.
- 보통은 노트북이 `mini_project/` 와 같은 디렉터리에 있다고 가정합니다. 만약 import 가 안 되면, 노트북 시작 셀에서 `sys.path` 에 `mini_project` 를 추가하세요.

```python
import sys; sys.path.insert(0, "mini_project")
```

In [ ]:
import sys
sys.path.insert(0, "mini_project")

from app.prompts.demo_questions import DEMO_QUESTIONS
import httpx

for i, q in enumerate(DEMO_QUESTIONS, start=1):
    print(f"\n========== Q{i}. {q} ==========")
    with httpx.Client(timeout=60.0) as c:
        with c.stream(
            "POST",
            f"{SERVER_BASE}/chat/stream",
            json={"message": q},
        ) as r:
            for piece in r.iter_text():
                print(piece, end="", flush=True)
    print("\n--- end ---")

## 완료 기준 체크리스트

발표 전에 아래 항목을 모두 확인하세요.

- [ ] 서버가 `uvicorn app.main:app --reload` 로 정상 기동
- [ ] `/chat/stream` 으로 호출 시 토큰이 스트리밍으로 흐름
- [ ] System Prompt 가 답변에 반영됨 (도메인 톤이 드러남)
- [ ] 데모 질문 5개 모두 정상 응답 (본문 끝 `[DONE]`)
- [ ] 같은 질문 두 번째 호출 시 서버 로그에 `cache HIT` 표시
- [ ] 발표 슬라이드 5장 작성 완료